In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.metrics import mean_squared_error as sklearn_mse ,r2_score

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df["Delivery_Time"].hist()

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID",axis=1)

In [ ]:
# Task 2: Write your code here:
df = df.dropna()

In [ ]:
# Task 3: Write your code here:
print("Number of duplecated: ",df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
print(categorical_cols)
# Label Enoder for columns need order
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])




In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

standard_scaler = StandardScaler() # Instantiate StandardScaler

featcher_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']

df[featcher_cols] = standard_scaler.fit_transform(df[featcher_cols])


In [ ]:
# Task 6: Write your code here:
df["Delivery_Time"].hist()
# not impelence

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time",axis=1)
y = df["Delivery_Time"]

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Task 2
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

model = RandomForestRegressor(100)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn model
  print(f"Training...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)

  print("MSE:",np.mean(lr_mse))
  print("RMSE:",np.mean(lr_rmse))
  print("R2:",np.mean(lr_mse))


In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Random Forest Regressor'] = model.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name}")
  ax.set_xlabel("feature importances")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = pd.DataFrame(model.predict(X_test))
y_pred.hist()

In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold



model1 = RandomForestRegressor(n_estimators=200)
model2 = CatBoostRegressor(verbose=0)

# Storage for results
all_results = {}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn model
  print(f"Training...")
  model1.fit(X_train, y_train) # train
  model2.fit(X_train, y_train)

  y_pred1 = model1.predict(X_test) # validate
  y_pred2 = model2.predict(X_test) # validate

  y_pred = y_pred1 + y_pred2 / 2
  mse = sklearn_mse(y_pred,y_test)
  print("MSE:",mse)
